<a href="https://colab.research.google.com/github/skywalk163/2020AICITY_Code_From_Top_Teams/blob/master/deepseekr1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 开始学习

In [5]:
%%writefile light_dataset.jsonl

Writing light_dataset.jsonl


In [ ]:
# 1. 优先通过 pip 官方源安装 unsloth (无需走 git clone)
!pip install unsloth

# 2. 安装配套依赖 (注意修正 accelerate 的拼写)
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.6 MB/s eta 0:00:00
  Using cached xformers-0.0.26.post1.tar.gz (4.1 MB)
  Preparing metadata (setup.py) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 34.1 MB/s eta 0:00:00


In [ ]:
# 在脚本开头添加：挂载 Google 云端硬盘
from google.colab import drive
drive.mount('/content/drive')

# 保存数据集或模型时，直接存入 Drive 目录：
# output_dir = "/content/drive/MyDrive/light_model_output"

In [ ]:
import json
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. 载入模型
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 2. 配置 LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 3. 读取 JSONL 数据集 (直接贴入或读取上传的文件)
data = []
with open("light_dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

dataset = Dataset.from_list(data)

# 格式化数据为 Qwen/DeepSeek 对话格式
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = f"<|im_start|>system\n{messages[0]['content']}<|im_end|>\n<|im_start|>user\n{messages[1]['content']}<|im_end|>\n<|im_start|>assistant\n{messages[2]['content']}<|im_end|>"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

# 4. 配置训练参数 (针对 Colab T4 单卡优化)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,      # 适配 Colab 显存
        gradient_accumulation_steps = 8,      # 保持总 Batch Size = 8
        warmup_steps = 5,
        max_steps = 60,                        # 50条数据跑 60 步
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# 5. 开始微调
trainer.train()

# 6. 保存微调权重至本地或 Google Drive
model.save_pretrained("light_lora_model")
tokenizer.save_pretrained("light_lora_model")

In [ ]:
# =====================================================================
# 1. 自动安装 Unsloth 和微调必备依赖环境（针对 Kaggle 环境深度优化）
# =====================================================================
import os
import torch

# 检查显卡驱动环境，自动安装适配的 Unsloth 版本
gpu_stats = torch.cuda.get_device_properties(0)
print(f"检测到正在使用显卡: {gpu_stats.name}")

!pip install --no-deps "unsloth[colab-new] @ git+https://github.com"
!pip install --no-deps trl peft transformers accelerate

# =====================================================================
# 2. 载入未损耗思考链的 DeepSeek-R1-1.5B 蒸馏模型
# =====================================================================
from unsloth import FastLanguageModel

max_seq_length = 2048  # 支持最高 2048 长度的光明语言上下文
dtype = None           # 自动检测当前显卡支持的精度 (Float16 或 Bfloat16)
load_in_4bit = True    # 开启 4-bit 量化，极低显存占用

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# =====================================================================
# 3. 配置针对《光明语言 (LightLang)》的 LoRA 核心微调参数
# =====================================================================
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                # LoRA 秩，1.5B模型 16-32 最佳
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # 全量门控网络对齐
    lora_alpha = 16,
    lora_dropout = 0,      # 0 经过优化性能最高
    bias = "none",
    use_gradient_checkpointing = "unsloth", # 使用 Unsloth 极速显存回收技术
    random_state = 3407,
    max_seq_length = max_seq_length,
)

# =====================================================================
# 4. 《光明语言》专用 R1 结构化提示词模板与数据清洗
# =====================================================================
from datasets import load_dataset

# 强迫模型在吐出代码前，必须先在 <think> 里用中文解构 LightLang 的语法树
light_prompt = """以下是关于编程语言或代码实现的需求，请先在 <think> 标签中思考如何使用光明语言（LightLang）实现，再给出最终的代码。

### 需求描述:
{}

### 思考与代码实现:
<think>
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]  # 注意：确保你的 jsonl 中 output 里含有类似 "思考... </think> 代码..." 的结构
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = light_prompt.format(instruction, output)
        texts.append(text)
    return { "text" : texts, }

# 💡 请在此处上传或指定你光明语言的数据集路径 (sft_dataset.jsonl)
# 你可以通过 Kaggle 右上角 "Data" -> "Upload" 上传你的数据集，然后复制其路径到下方
dataset_path = "sft_dataset.jsonl"

if os.path.exists(dataset_path):
    dataset = load_dataset("json", data_files=dataset_path, split="train")
    dataset = dataset.map(formatting_prompts_func, batched = True)
    print("✅ 数据集载入并格式化成功！")
else:
    print(f"⚠️ 未检测到数据集文件 {dataset_path}，请先在左侧/右侧面板上传该文件。当前将使用 Dummy 虚拟测试数据继续运行逻辑。")
    # 创建一个虚拟的数据集防止脚本报错
    from datasets import Dataset
    dummy_data = {"instruction": ["写一个光明语言函数"], "output": ["如何实现...</think>段落 测试接收: 打印 \"1\""]}
    dataset = Dataset.from_dict(dummy_data).map(formatting_prompts_func, batched = True)

# =====================================================================
# 5. 配置训练超参数并开启微调
# =====================================================================
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # 对短文本/代码补全更友好
    training_args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,                # 依据你的数据集行数调整，800条数据建议改为 200-300
        learning_rate = 2e-5,          # 小学习率，全力保护 R1 原生逻辑底子
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",          # 8bit 优化器，压榨每一卡显存
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 开始微调模型...")
trainer_stats = trainer.train()
print("🎉 微调完成！")

# =====================================================================
# 6. 一键将微调后的模型打包并导出为老 CPU 最爱的 GGUF 量化格式
# =====================================================================
print("📦 正在将模型转换为老旧 CPU 专属的 GGUF (Q4_K_M) 格式...")
# 该操作会自动将 LoRA 权重合并入基座，并利用 llama.cpp 核心将其量化为 4位精度
model.save_pretrained_gguf(
    "lightlang_r1_qwen_1.5b_q4",
    tokenizer,
    quantization_method = "q4_k_m"
)
print("🎯 转换成功！你现在可以在右侧面板的 Output 文件夹中直接下载 lightlang_r1_qwen_1.5b_q4-unsloth.Q4_K_M.gguf 文件了！")
